### Задание 1.

Решите задачу распознавания лиц с помощью SVM с ядром. Попробуйте различные ядра: 'poly', 'rbf', 'sigmoid'.

Подберите гиперпараметры по кросс-валидации.

SVM с каким ядром дал лучший результат?

In [44]:
#импортируем датасет с лицами

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn.datasets
from sklearn.datasets import fetch_lfw_people
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA #Principal Components Analysis
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

faces = fetch_lfw_people(min_faces_per_person=60)
print(faces.target_names)
print(faces.images.shape)

In [69]:
#уменьшение размерности
pca = PCA(n_components=150, svd_solver='randomized', whiten=True, random_state=42)
Xtrain, Xtest, ytrain, ytest = train_test_split(faces.data, faces.target, test_size=0.3,
                                                random_state=42)

In [70]:
#создаём три модели с тремя ядрами
#можно было бы сначала заразмерить, но у меня вылезла ошибка - я решила не рисковать
svc_p = SVC(kernel='poly')
svc_r = SVC(kernel='rbf')
svc_s = SVC(kernel='sigmoid')
model1 = make_pipeline(pca, svc_p)
model2 = make_pipeline(pca, svc_r)
model3 = make_pipeline(pca, svc_s)
print(model1, model2, model3)

Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC(kernel='poly'))]) Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC())]) Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC(kernel='sigmoid'))])


In [71]:
#подберём параметр C
models = [model1, model2, model3]
param_grid = {'svc__C': [1, 5, 10, 50]}
for i in models:
  grid = GridSearchCV(i, param_grid)
  grid.fit(Xtrain, ytrain)
  print(grid.best_params_, i)

{'svc__C': 50} Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC(kernel='poly'))])
{'svc__C': 5} Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC())])
{'svc__C': 1} Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC(kernel='sigmoid'))])


In [72]:
#при дефолтном test size лучше справляется rbf (accuracy = 0.83), но сигмоида выдаёт 0,
#при увеличении test size ухудшается poly (с 0.7 до 0.36 где-то), но сигмоида вырастает до 0.77
#rbf работает и так и так

svc_p, svc_r, svc_s = SVC(C=50, kernel='poly'), SVC(C=5, kernel='rbf'), SVC(C = 1, kernel='sigmoid')
model1, model2, model3 = make_pipeline(pca, svc_p), make_pipeline(pca, svc_r), make_pipeline(pca, svc_s)
for i in models:
  i.fit(Xtrain, ytrain)
  yfit = i.predict(Xtest)
  print(i)
  print(classification_report(ytest, yfit,
                            target_names=faces.target_names))
  accuracy_score(yfit,ytest)

Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC(kernel='poly'))])
                   precision    recall  f1-score   support

     Ariel Sharon       0.00      0.00      0.00        17
     Colin Powell       0.00      0.00      0.00        84
  Donald Rumsfeld       0.00      0.00      0.00        36
    George W Bush       0.36      1.00      0.53       146
Gerhard Schroeder       0.00      0.00      0.00        28
      Hugo Chavez       0.00      0.00      0.00        27
Junichiro Koizumi       0.00      0.00      0.00        16
       Tony Blair       0.00      0.00      0.00        51

         accuracy                           0.36       405
        macro avg       0.05      0.12      0.07       405
     weighted avg       0.13      0.36      0.19       405



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
                     whiten=True)),
                ('svc', SVC())])
                   precision    recall  f1-score   support

     Ariel Sharon       1.00      0.47      0.64        17
     Colin Powell       0.90      0.82      0.86        84
  Donald Rumsfeld       0.94      0.47      0.63        36
    George W Bush       0.59      0.98      0.73       146
Gerhard Schroeder       0.93      0.50      0.65        28
      Hugo Chavez       1.00      0.19      0.31        27
Junichiro Koizumi       1.00      0.69      0.81        16
       Tony Blair       0.93      0.49      0.64        51

         accuracy                           0.72       405
        macro avg       0.91      0.58      0.66       405
     weighted avg       0.81      0.72      0.70       405

Pipeline(steps=[('pca',
                 PCA(n_components=150, random_state=42, svd_solver='randomized',
          

### Задание 2.

Решите задачу распознавания лиц с помощью логистической регрессии (она также поддерживает опцию class_weight='balanced'):

1) Объявите модель, состоящую из pipeline(pca,logistic regression)

2) Подберите по сетке параметр C логистической регрессии (с помощью GridSearch)

3) Обучите модель на тренировочных данных и выведите наилучшие параметры модели

Какое качество показала эта модель?

In [33]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression(class_weight='balanced', random_state=13)
model4 = make_pipeline(pca, lr)
param_grid = {"logisticregression__C": [1, 5, 50, 100]}
grid = GridSearchCV(model4, param_grid)
grid.fit(Xtrain, ytrain)
print(grid.best_params_)

NameError: name 'pca' is not defined

In [63]:
#модель показала достаточно высокую точность - 0.82
model4 = grid.best_estimator_
model4.fit(Xtrain, ytrain)
yfit = model4.predict(Xtest)
print(classification_report(ytest, yfit,
                            target_names=faces.target_names))
accuracy_score(yfit,ytest)

                   precision    recall  f1-score   support

     Ariel Sharon       0.63      0.80      0.71        15
     Colin Powell       0.85      0.88      0.86        68
  Donald Rumsfeld       0.69      0.71      0.70        31
    George W Bush       0.93      0.79      0.85       126
Gerhard Schroeder       0.71      0.74      0.72        23
      Hugo Chavez       0.88      0.70      0.78        20
Junichiro Koizumi       0.86      1.00      0.92        12
       Tony Blair       0.71      0.93      0.80        42

         accuracy                           0.82       337
        macro avg       0.78      0.82      0.79       337
     weighted avg       0.83      0.82      0.82       337



0.8160237388724035

### Задание 3.

Разбалловка:

- 5 баллов: обучили один алгоритм и погридсерчили
- 10 баллов: попробовали обучить два и более алгоритмов, погридсерчили

Поработайте с датасетом winequalityN (целевая переменная - quality). Поэкспериментируйте с алгоритмами классификации, попробуйте подобрать гиперпараметры для них.

In [36]:
import pandas as pd

data = pd.read_csv('winequalityN.csv')
data.head()

,type,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,white,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,white,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,white,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,white,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [37]:
data['quality'].value_counts()

quality
6    2836
5    2138
7    1079
4     216
8     193
3      30
9       5
Name: count, dtype: int64

In [20]:
data.info() #почти все числовое

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   type                  6497 non-null   object 
 1   fixed acidity         6487 non-null   float64
 2   volatile acidity      6489 non-null   float64
 3   citric acid           6494 non-null   float64
 4   residual sugar        6495 non-null   float64
 5   chlorides             6495 non-null   float64
 6   free sulfur dioxide   6497 non-null   float64
 7   total sulfur dioxide  6497 non-null   float64
 8   density               6497 non-null   float64
 9   pH                    6488 non-null   float64
 10  sulphates             6493 non-null   float64
 11  alcohol               6497 non-null   float64
 12  quality               6497 non-null   int64  
dtypes: float64(11), int64(1), object(1)
memory usage: 660.0+ KB


In [21]:
data.isna().sum()

type                     0
fixed acidity           10
volatile acidity         8
citric acid              3
residual sugar           2
chlorides                2
free sulfur dioxide      0
total sulfur dioxide     0
density                  0
pH                       9
sulphates                4
alcohol                  0
quality                  0
dtype: int64

In [27]:
data['type'].unique() #либо белое, либо красное - можно заменить на 0/1
data['type'] = data['type'].apply(lambda x: 1 if x== "red" else 0)
data.dropna(how='any', inplace=True) #заполненных строчек достаточно - просто удалим незаполненное
data['type'].value_counts()

type
0    4870
1    1593
Name: count, dtype: int64

In [31]:
X = data.drop('quality', axis=1)
y = data['quality']
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, random_state=42)

In [65]:
#посмотрим деревья решений
dt = DecisionTreeClassifier(random_state=13)
dt.fit(Xtrain, ytrain)
y_pred_dt = dt.predict(Xtest)
print(classification_report(ytest, y_pred_dt))
print(f"Decision tree accuracy: {accuracy_score(y_pred_dt, ytest):.2f}")

              precision    recall  f1-score   support

           3       0.00      0.00      0.00         6
           4       0.16      0.16      0.16        58
           5       0.64      0.61      0.62       540
           6       0.62      0.60      0.61       686
           7       0.49      0.55      0.52       274
           8       0.33      0.37      0.35        51
           9       0.00      0.00      0.00         1

    accuracy                           0.57      1616
   macro avg       0.32      0.33      0.32      1616
weighted avg       0.57      0.57      0.57      1616

Decision tree accuracy: 0.57


In [64]:
#посмотрим регрессию
#деревья решений справились немного лучше, чем регрессия
lr2 = LogisticRegression()
param_grid = {'C': [1, 5, 50, 100]}
grid = GridSearchCV(estimator = lr2, param_grid=param_grid)
grid.fit(Xtrain, ytrain)
print(grid.best_params_)
model5 = grid.best_estimator_
model5.fit(Xtrain, ytrain)
yfit = model5.predict(Xtest)
print(classification_report(ytest, yfit))
print(f'Logistic Regression accuracy: {accuracy_score(yfit,ytest)}')

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMI

{'C': 1}
              precision    recall  f1-score   support

           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00        58
           5       0.52      0.36      0.43       540
           6       0.44      0.79      0.56       686
           7       0.00      0.00      0.00       274
           8       0.00      0.00      0.00        51
           9       0.00      0.00      0.00         1

    accuracy                           0.46      1616
   macro avg       0.14      0.16      0.14      1616
weighted avg       0.36      0.46      0.38      1616

Logistic Regression accuracy: 0.4560643564356436


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sklearn/metr